# Détection de fractures osseuses – FracAtlas
**Projet IA02 – Classification non linéaire grâce à l'IA avancée**

**Étudiants :** Yves CHEKOUA & Mohamed Mehdi TRABELSSI
**Promotion :** SN2 – Université de Technologie de Troyes


In [ ]:
# ── Bibliothèques de base ──────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import warnings
warnings.filterwarnings('ignore')

# ── Machine Learning classique ────────────────────────────────────
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, roc_auc_score, roc_curve)

# ── Deep Learning ─────────────────────────────────────────────────
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.datasets import cifar10

print(f"TensorFlow : {tf.__version__}")
print(f"GPU disponible : {len(tf.config.list_physical_devices('GPU')) > 0}")


---
# Partie 2 : Détection de fractures – FracAtlas
**Problème :** classification binaire d'images radiographiques (Fracturée / Non fracturée)
**Enjeu médical :** minimiser les faux négatifs (fractures non détectées)
**Défi :** déséquilibre de classes → ratio 1:4.7 (717 fracturées vs 3 366 non-fracturées)


In [ ]:
import os
import cv2
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve

# Chemin vers le dataset FracAtlas
# ⚠️ Adapter ce chemin selon votre environnement
DATA_DIR = r'FracAtlas/images'   # Colab : '/content/FracAtlas/images'

CLASSES_P2 = ['Non_fracturee', 'Fracturee']
IMG_ML     = 32   # résolution pour RF et SVM (rapide)
IMG_CNN_P2 = 96   # résolution pour MobileNetV2

# Vérification du dataset
for folder in ['Non_fractured', 'Fractured']:
    path = os.path.join(DATA_DIR, folder)
    if os.path.exists(path):
        print(f"{folder}: {len(os.listdir(path))} images")
    else:
        print(f"⚠️  Dossier introuvable : {path}")


## 1. Analyse et chargement du dataset FracAtlas

In [ ]:
def load_dataset(data_dir, img_size, max_per_class=None):
    """Charge les images depuis les dossiers Non_fractured / Fractured."""
    X, y = [], []
    for label, folder in enumerate(['Non_fractured', 'Fractured']):
        folder_path = os.path.join(data_dir, folder)
        files = sorted([f for f in os.listdir(folder_path)
                        if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        if max_per_class:
            files = files[:max_per_class]
        for fname in files:
            img = cv2.imread(os.path.join(folder_path, fname))
            if img is None:
                continue
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (img_size, img_size))
            X.append(img)
            y.append(label)
    return np.array(X, dtype='uint8'), np.array(y)

# Chargement du dataset complet pour RF et SVM (717 fracturées / 3366 non fracturées)
print("Chargement images 32×32 (RF et SVM) — dataset complet...")
t0 = time.time()
X_ml, y_ml = load_dataset(DATA_DIR, IMG_ML)
print(f"  {X_ml.shape}  en {time.time()-t0:.1f} s")

# Chargement du dataset complet pour CNN
print("Chargement images 96×96 (CNN) — dataset complet...")
t0 = time.time()
X_cnn_p2, y_cnn_p2 = load_dataset(DATA_DIR, IMG_CNN_P2)
print(f"  {X_cnn_p2.shape}  en {time.time()-t0:.1f} s")


In [ ]:
# Distribution des classes
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = [(y_ml==0).sum(), (y_ml==1).sum()]
axes[0].bar(CLASSES_P2, counts, color=['steelblue', 'tomato'])
axes[0].set_title('Distribution – dataset équilibré (ML)')
for i, v in enumerate(counts):
    axes[0].text(i, v+5, str(v), ha='center', fontweight='bold')

# Exemples d'images
axes[1].axis('off')
fig2, axs = plt.subplots(2, 5, figsize=(13, 6))
for label in [0, 1]:
    idxs = np.where(y_ml == label)[0][:5]
    for j, idx in enumerate(idxs):
        axs[label][j].imshow(X_ml[idx])
        axs[label][j].set_title(CLASSES_P2[label], fontsize=8)
        axs[label][j].axis('off')
plt.suptitle('Exemples de radiographies FracAtlas')
plt.tight_layout(); plt.show()


## 2. Préparation des données
**Dataset complet** utilisé pour RF, SVM et CNN (4 083 images : 717 fracturées, 3 366 non fracturées).
Le déséquilibre de classes est compensé par `class_weight='balanced'` (RF, SVM) et une pondération
manuelle `{0: 1.0, 1: 4.7}` (CNN), plutôt que par sous-échantillonnage.


In [ ]:
# Split stratifié 80/20 – conservation des proportions de classes
X_ml_tr, X_ml_te, y_ml_tr, y_ml_te = train_test_split(
    X_ml, y_ml, test_size=0.2, random_state=42, stratify=y_ml
)

# Normalisation + aplatissement pour RF/SVM
X_tr_flat = X_ml_tr.reshape(len(X_ml_tr), -1).astype('float32') / 255.0
X_te_flat = X_ml_te.reshape(len(X_ml_te), -1).astype('float32') / 255.0

print(f"Train ML : {X_tr_flat.shape}  |  Test ML : {X_te_flat.shape}")
print(f"  Train – Non-frac: {(y_ml_tr==0).sum()}  |  Frac: {(y_ml_tr==1).sum()}")
print(f"  Test  – Non-frac: {(y_ml_te==0).sum()}  |  Frac: {(y_ml_te==1).sum()}")


## 3. Augmentation des données
L'augmentation génère synthétiquement de nouveaux exemples, essentiel pour la classe minoritaire.
Elle est intégrée dans le pipeline CNN (active uniquement en entraînement).


In [ ]:
# Pipeline d'augmentation intégré au CNN
augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal_and_vertical'),  # fractures peuvent être des deux côtés
    layers.RandomRotation(0.15),                   # radiographies légèrement inclinées
    layers.RandomZoom(0.15),                       # différentes distances de prise de vue
    layers.RandomContrast(0.2),                    # variation d'exposition radiographique
], name='augmentation_fractures')

# Visualisation de l'effet de l'augmentation
X_cnn_norm_temp = X_cnn_p2[:1].astype('float32') / 127.5 - 1.0
frac_idxs = np.where(y_cnn_p2 == 1)[0]
sample = (X_cnn_p2[frac_idxs[0]:frac_idxs[0]+1].astype('float32') / 127.5 - 1.0)

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
axes.flat[0].imshow((sample[0]+1)/2)
axes.flat[0].set_title('Original', fontweight='bold')
axes.flat[0].axis('off')
for ax in list(axes.flat)[1:]:
    aug = augmentation(sample, training=True)[0].numpy()
    ax.imshow(np.clip((aug+1)/2, 0, 1))
    ax.axis('off')
plt.suptitle('Augmentation de données – Radiographie fracturée')
plt.tight_layout(); plt.show()


## 4. Algorithme 1 : Random Forest (fractures)
Métrique d'optimisation : **F1-score** (plus pertinent que l'accuracy sur données déséquilibrées).


In [ ]:
param_grid_rf2 = {
    'n_estimators': [100, 200],
    'max_depth':    [10, 20, None],
    'max_features': ['sqrt', 'log2']
}

print("GridSearchCV RF fractures (~1-2 min)...")
t0 = time.time()
gs_rf_p2 = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1, class_weight='balanced'),
    param_grid_rf2, cv=3, scoring='f1', verbose=1, n_jobs=-1
)
gs_rf_p2.fit(X_tr_flat, y_ml_tr)
print(f"Temps : {time.time()-t0:.0f} s  |  Meilleurs params : {gs_rf_p2.best_params_}")

rf_p2 = gs_rf_p2.best_estimator_
y_pred_rf2 = rf_p2.predict(X_te_flat)
y_prob_rf2 = rf_p2.predict_proba(X_te_flat)[:, 1]
acc_rf2  = accuracy_score(y_ml_te, y_pred_rf2)
auc_rf2  = roc_auc_score(y_ml_te, y_prob_rf2)

print(f"\nRandom Forest – Exactitude : {acc_rf2*100:.2f}%  |  AUC : {auc_rf2:.4f}")
print()
print(classification_report(y_ml_te, y_pred_rf2, target_names=CLASSES_P2))

plt.figure(figsize=(5, 4))
sns.heatmap(confusion_matrix(y_ml_te, y_pred_rf2), annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASSES_P2, yticklabels=CLASSES_P2)
plt.title(f'RF – Matrice de confusion (acc={acc_rf2*100:.1f}%)')
plt.xlabel('Prédit'); plt.ylabel('Réel')
plt.tight_layout(); plt.show()


## 5. Algorithme 2 : SVM (fractures)
**StandardScaler** obligatoire avant le SVM (centrage-réduction des features).


In [ ]:
# Standardisation des features (indispensable pour le SVM)
scaler_p2 = StandardScaler()
X_tr_sc = scaler_p2.fit_transform(X_tr_flat)
X_te_sc = scaler_p2.transform(X_te_flat)

param_grid_svm2 = {'C': [0.1, 1, 10], 'kernel': ['rbf', 'linear']}

print("GridSearchCV SVM fractures (~2-4 min)...")
t0 = time.time()
gs_svm_p2 = GridSearchCV(
    SVC(probability=True, random_state=42, class_weight='balanced'),
    param_grid_svm2, cv=3, scoring='f1', verbose=1, n_jobs=-1
)
gs_svm_p2.fit(X_tr_sc, y_ml_tr)
print(f"Temps : {time.time()-t0:.0f} s  |  Meilleurs params : {gs_svm_p2.best_params_}")

svm_p2 = gs_svm_p2.best_estimator_
y_pred_svm2 = svm_p2.predict(X_te_sc)
y_prob_svm2 = svm_p2.predict_proba(X_te_sc)[:, 1]
acc_svm2 = accuracy_score(y_ml_te, y_pred_svm2)
auc_svm2 = roc_auc_score(y_ml_te, y_prob_svm2)

print(f"\nSVM – Exactitude : {acc_svm2*100:.2f}%  |  AUC : {auc_svm2:.4f}")
print()
print(classification_report(y_ml_te, y_pred_svm2, target_names=CLASSES_P2))

plt.figure(figsize=(5, 4))
sns.heatmap(confusion_matrix(y_ml_te, y_pred_svm2), annot=True, fmt='d', cmap='Oranges',
            xticklabels=CLASSES_P2, yticklabels=CLASSES_P2)
plt.title(f'SVM – Matrice de confusion (acc={acc_svm2*100:.1f}%)')
plt.xlabel('Prédit'); plt.ylabel('Réel')
plt.tight_layout(); plt.show()


## 6. Algorithmes 3 & 4 : CNN Transfer Learning MobileNetV2 (fractures)
Même approche Freeze que la Partie 1, adaptée à la **classification binaire**.
Deux variantes sont comparées : **sans augmentation** (Algorithme 3) et **avec augmentation**
(Algorithme 4), afin de mesurer l'impact réel de l'augmentation de données sur ce dataset.


In [ ]:
# Split CNN
X_cnn_tr, X_cnn_te, y_cnn_tr, y_cnn_te = train_test_split(
    X_cnn_p2, y_cnn_p2, test_size=0.2, random_state=42, stratify=y_cnn_p2
)
# Normalisation MobileNetV2 : [-1, 1]
X_cnn_tr = X_cnn_tr.astype('float32') / 127.5 - 1.0
X_cnn_te = X_cnn_te.astype('float32') / 127.5 - 1.0

# Poids de classe pour compenser le déséquilibre résiduel
n_neg = (y_cnn_tr == 0).sum()
n_pos = (y_cnn_tr == 1).sum()
class_w = {0: 1.0, 1: n_neg / n_pos}

print(f"Train CNN : {X_cnn_tr.shape}  |  Test CNN : {X_cnn_te.shape}")
print(f"Poids classe Fracturée : {class_w[1]:.2f}")


### 6.1 Algorithme 3 : CNN MobileNetV2 (sans augmentation) — meilleur modèle

In [ ]:
# Construction du modèle SANS augmentation
base_p2 = MobileNetV2(weights='imagenet', include_top=False,
                      input_shape=(IMG_CNN_P2, IMG_CNN_P2, 3))
for layer in base_p2.layers:
    layer.trainable = False

inp = layers.Input(shape=(IMG_CNN_P2, IMG_CNN_P2, 3))
x   = base_p2(inp, training=False)               # extraction de features (gelée)
x   = layers.GlobalAveragePooling2D()(x)
x   = layers.Dense(64, activation='relu')(x)
x   = layers.Dropout(0.4)(x)
out = layers.Dense(1, activation='sigmoid')(x)   # sortie binaire

model_p2 = models.Model(inputs=inp, outputs=out, name='CNN_Fracture_MobileNetV2')
print(f"Paramètres entraînables : {sum(w.numpy().size for w in model_p2.trainable_weights):,}")


In [ ]:
model_p2.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)
cb_p2 = [
    EarlyStopping(monitor='val_auc', patience=6, restore_best_weights=True,
                  mode='max', verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1)
]

print("Phase 1 – Freeze (tête FC uniquement)...")
t0 = time.time()
hist_p2_freeze = model_p2.fit(
    X_cnn_tr, y_cnn_tr,
    epochs=20, batch_size=32, validation_split=0.1,
    class_weight=class_w, callbacks=cb_p2, verbose=1
)
t_p2_freeze = time.time() - t0
print(f"Phase 1 terminée en {t_p2_freeze/60:.1f} min")


In [ ]:
# Dégeler les 30 dernières couches pour le fine-tuning
for layer in base_p2.layers[-30:]:
    layer.trainable = True

model_p2.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),  # lr très faible : ne pas écraser ImageNet
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

print("Phase 2 – Fine-tuning (30 dernières couches)...")
t0 = time.time()
hist_p2_ft = model_p2.fit(
    X_cnn_tr, y_cnn_tr,
    epochs=10, batch_size=32, validation_split=0.1,
    class_weight=class_w, callbacks=cb_p2, verbose=1
)
t_p2_ft = time.time() - t0
t_p2_total = t_p2_freeze + t_p2_ft
print(f"Temps total CNN fractures (sans augmentation) : {t_p2_total/60:.1f} min")


In [ ]:
# Évaluation – CNN sans augmentation (meilleur modèle de la Partie 2)
y_prob_cnn2 = model_p2.predict(X_cnn_te, verbose=0).flatten()
y_pred_cnn2 = (y_prob_cnn2 >= 0.5).astype(int)
acc_cnn2_p2 = accuracy_score(y_cnn_te, y_pred_cnn2)
auc_cnn2    = roc_auc_score(y_cnn_te, y_prob_cnn2)

print(f"CNN MobileNetV2 (sans augmentation) – Exactitude : {acc_cnn2_p2*100:.2f}%  |  AUC : {auc_cnn2:.4f}")
print()
print(classification_report(y_cnn_te, y_pred_cnn2, target_names=CLASSES_P2))

plt.figure(figsize=(5, 4))
sns.heatmap(confusion_matrix(y_cnn_te, y_pred_cnn2), annot=True, fmt='d', cmap='Greens',
            xticklabels=CLASSES_P2, yticklabels=CLASSES_P2)
plt.title(f'CNN (sans augmentation) – Matrice de confusion (acc={acc_cnn2_p2*100:.1f}%)')
plt.xlabel('Prédit'); plt.ylabel('Réel')
plt.tight_layout(); plt.show()

# Courbes d'apprentissage
acc_p2 = hist_p2_freeze.history['auc']    + hist_p2_ft.history['auc']
val_p2 = hist_p2_freeze.history['val_auc']+ hist_p2_ft.history['val_auc']
ep_s   = len(hist_p2_freeze.history['auc'])
plt.figure(figsize=(8, 4))
plt.plot(acc_p2, label='Train AUC', linewidth=2)
plt.plot(val_p2, label='Val AUC', linewidth=2)
plt.axvline(ep_s-1, color='gray', linestyle='--', label='Début fine-tuning')
plt.title('CNN MobileNetV2 (sans augmentation) – AUC (Fractures)')
plt.xlabel('Époque'); plt.legend(); plt.grid(True)
plt.tight_layout(); plt.show()


### 6.2 Algorithme 4 : CNN MobileNetV2 (avec augmentation) — contre-exemple
Même architecture, avec la couche d'augmentation (incluant `RandomFlip` vertical) intégrée
en amont du réseau.


In [ ]:
# Construction du modèle AVEC augmentation intégrée
base_p2_aug = MobileNetV2(weights='imagenet', include_top=False,
                          input_shape=(IMG_CNN_P2, IMG_CNN_P2, 3))
for layer in base_p2_aug.layers:
    layer.trainable = False

inp_aug = layers.Input(shape=(IMG_CNN_P2, IMG_CNN_P2, 3))
x   = augmentation(inp_aug)                       # augmentation en début de pipeline
x   = base_p2_aug(x, training=False)              # extraction de features (gelée)
x   = layers.GlobalAveragePooling2D()(x)
x   = layers.Dense(64, activation='relu')(x)
x   = layers.Dropout(0.4)(x)
out_aug = layers.Dense(1, activation='sigmoid')(x)

model_p2_aug = models.Model(inputs=inp_aug, outputs=out_aug, name='CNN_Fracture_MobileNetV2_Aug')
print(f"Paramètres entraînables : {sum(w.numpy().size for w in model_p2_aug.trainable_weights):,}")


In [ ]:
model_p2_aug.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)
cb_p2_aug = [
    EarlyStopping(monitor='val_auc', patience=6, restore_best_weights=True,
                  mode='max', verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1)
]

print("Phase 1 – Freeze (avec augmentation)...")
t0 = time.time()
hist_p2_aug_freeze = model_p2_aug.fit(
    X_cnn_tr, y_cnn_tr,
    epochs=20, batch_size=32, validation_split=0.1,
    class_weight=class_w, callbacks=cb_p2_aug, verbose=1
)
print(f"Phase 1 terminée en {(time.time()-t0)/60:.1f} min")


In [ ]:
for layer in base_p2_aug.layers[-30:]:
    layer.trainable = True

model_p2_aug.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

print("Phase 2 – Fine-tuning (avec augmentation)...")
t0 = time.time()
hist_p2_aug_ft = model_p2_aug.fit(
    X_cnn_tr, y_cnn_tr,
    epochs=10, batch_size=32, validation_split=0.1,
    class_weight=class_w, callbacks=cb_p2_aug, verbose=1
)
print(f"Phase 2 terminée en {(time.time()-t0)/60:.1f} min")


In [ ]:
# Évaluation – CNN avec augmentation
y_prob_cnn2_aug = model_p2_aug.predict(X_cnn_te, verbose=0).flatten()
y_pred_cnn2_aug = (y_prob_cnn2_aug >= 0.5).astype(int)
acc_cnn2_aug = accuracy_score(y_cnn_te, y_pred_cnn2_aug)
auc_cnn2_aug = roc_auc_score(y_cnn_te, y_prob_cnn2_aug)

print(f"CNN MobileNetV2 (avec augmentation) – Exactitude : {acc_cnn2_aug*100:.2f}%  |  AUC : {auc_cnn2_aug:.4f}")
print()
print(classification_report(y_cnn_te, y_pred_cnn2_aug, target_names=CLASSES_P2))

plt.figure(figsize=(5, 4))
sns.heatmap(confusion_matrix(y_cnn_te, y_pred_cnn2_aug), annot=True, fmt='d', cmap='Reds',
            xticklabels=CLASSES_P2, yticklabels=CLASSES_P2)
plt.title(f'CNN (avec augmentation) – Matrice de confusion (acc={acc_cnn2_aug*100:.1f}%)')
plt.xlabel('Prédit'); plt.ylabel('Réel')
plt.tight_layout(); plt.show()

print()
print("Observation : l'augmentation dégrade ici la performance (-10,8 pts d'exactitude).")
print("Le RandomFlip vertical déforme la sémantique anatomique des radiographies —")
print("une augmentation mal choisie peut nuire plutôt qu'aider.")


## 7. Comparaison générale – Partie 2

In [ ]:
# Courbes ROC des 4 algorithmes
plt.figure(figsize=(8, 6))
for name, y_true_r, y_prob_r, auc_v in [
    ('Random Forest',                   y_ml_te,  y_prob_rf2,      auc_rf2),
    ('SVM',                             y_ml_te,  y_prob_svm2,     auc_svm2),
    ('CNN MobileNetV2',                 y_cnn_te, y_prob_cnn2,     auc_cnn2),
    ('CNN MobileNetV2 (Augmentation)',  y_cnn_te, y_prob_cnn2_aug, auc_cnn2_aug),
]:
    fpr, tpr, _ = roc_curve(y_true_r, y_prob_r)
    plt.plot(fpr, tpr, linewidth=2, label=f'{name} (AUC={auc_v:.3f})')
plt.plot([0,1],[0,1],'k--', label='Aléatoire')
plt.xlabel('Taux de faux positifs')
plt.ylabel('Taux de vrais positifs')
plt.title('Courbes ROC – Détection de fractures')
plt.legend(); plt.grid(True)
plt.tight_layout(); plt.show()


In [ ]:
# Tableau comparatif Partie 2
print('═' * 58)
print(f'{"Algorithme":<28} {"Exactitude":>12} {"AUC-ROC":>10}')
print('─' * 58)
for name, acc, auc_v in [
    ('Random Forest',                  acc_rf2,     auc_rf2),
    ('SVM',                            acc_svm2,    auc_svm2),
    ('CNN MobileNetV2',                acc_cnn2_p2, auc_cnn2),
    ('CNN MobileNetV2 (Augmentation)', acc_cnn2_aug, auc_cnn2_aug),
]:
    print(f'{name:<28} {acc*100:>11.2f}% {auc_v:>10.4f}')
print('═' * 58)

# Graphique
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
noms = ['Random\nForest', 'SVM', 'CNN\nMobileNetV2', 'CNN +\nAugmentation']
accs2 = [acc_rf2, acc_svm2, acc_cnn2_p2, acc_cnn2_aug]
aucs  = [auc_rf2, auc_svm2, auc_cnn2, auc_cnn2_aug]
colors2 = ['steelblue', 'darkorange', 'seagreen', 'indianred']

axes[0].bar(noms, [a*100 for a in accs2], color=colors2)
axes[0].set_title('Exactitude – Fractures'); axes[0].set_ylabel('%')
for i, v in enumerate(accs2):
    axes[0].text(i, v*100+0.5, f'{v*100:.1f}%', ha='center', fontweight='bold')

axes[1].bar(noms, aucs, color=colors2)
axes[1].set_title('AUC-ROC – Fractures'); axes[1].set_ylim(0, 1)
for i, v in enumerate(aucs):
    axes[1].text(i, v+0.01, f'{v:.3f}', ha='center', fontweight='bold')

plt.tight_layout(); plt.show()


---
# Conclusion générale

## Partie 1 – CIFAR-10
| Modèle | Exactitude | Surapprentissage |
|--------|-----------|-----------------|
| SVM (RBF) | ~47% | – |
| Régression Logistique | ~40% | – |
| Random Forest | ~47% | Fort (99% train) |
| CNN Simple | ~71% | Modéré |
| CNN Intermédiaire | ~72% | Fort |
| CNN Avancé (Régularisé) | ~82% | Faible |
| **CNN Hybride MobileNetV2** | **~87%** | **Faible** |

## Partie 2 – Détection de fractures (dataset complet)
| Modèle | Exactitude | AUC-ROC |
|--------|-----------|---------|
| Random Forest | ~83,6% | ~0.730 |
| SVM | ~74,3% | ~0.745 |
| **CNN MobileNetV2 (sans augmentation)** | **~86,05%** | **~0.848** |
| CNN MobileNetV2 (avec augmentation) | ~75,3% | ~0.626 |

> Sur le dataset complet, le Transfer Learning (CNN MobileNetV2 sans augmentation) redevient le
> meilleur modèle, conformément à la Partie 1. L'augmentation de données dégrade ici les
> performances (-10,8 pts) : le `RandomFlip` vertical n'est pas adapté à la sémantique anatomique
> des radiographies.

## Enseignements
- Le **transfer learning** est l'approche la plus performante sur un dataset suffisamment grand (87,25% CIFAR-10, 86,05% FracAtlas)
- L'**augmentation de données n'est pas systématiquement bénéfique** : elle doit être choisie selon le domaine (le flip vertical nuit à la détection de fractures)
- La **régularisation** (Dropout, BatchNorm) est essentielle pour les CNN profonds
- Sur données déséquilibrées, privilégier l'**AUC-ROC** et le **F1-score** à l'accuracy
